# Field Shift · Punto 7: ordenar las rotaciones según lo que le importa al productor

**Qué ve el productor:** en "Next season" marca lo que le importa (menos riesgo, más ingreso, salud del suelo, ahorrar agua) y las opciones se ordenan según eso, con el porqué en plata:

> "With your priorities (less risk, more income), corn after soybeans comes first: in your 2 worst climate years it leaves about $464/acre, against $410 with your usual mix."

**No entrena un modelo nuevo y no toca el rinde.** Es una **capa de decisión** arriba del modelo B del punto 2: usa los escenarios de clima que ya trae `escenarios_rotacion.csv` (el rinde de cada opción con el clima de cada año 2009–2023) y los pasa a plata con el precio de NASS y el costo del ERS. Las prioridades no entran al modelo: el rinde es el mismo para todos los productores; lo que cambia es cómo se ordenan las opciones.

**Se muestra como "orden según tus prioridades", nunca como recomendación.**

**Cómo usarla:** correr todo de arriba hacia abajo, después de los puntos 2 y 3 y de `scripts/build_precios_costos.py`. Propuesta original: PR #37 (Diego); issue #41.

## 0. Librerías

- **pandas / numpy:** tablas y cuentas.
- **scikit-learn:** la regresión logística del paso 6 (pesos a partir de comparar de a pares).

In [1]:
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

## 1. Parámetros

- `FRACCION_PEORES`: qué parte de los años de clima cuenta como "años malos" para el riesgo. Con 0,10 y 15 años son **2 años** (se redondea hacia arriba con `math.ceil`; con `// 10` sería 1 solo año).
- `SUELO_REGLA`: la regla agronómica de salud del suelo que ya usa el front (rotar ayuda; repetir el cultivo no). No sale del modelo.
- `FACTOR_CONFIANZA`: a una opción con confianza **baja** (pocos años de rinde, o Missouri en rotado y continuo) se le achican los puntajes de ingreso y riesgo. La confianza media no se castiga: es la de **todas** las opciones rotado y continuo (el efecto de rotar se midió con cambios chicos), y castigarla favorecería siempre a seguir igual.
- `PESOS_EJEMPLO`: un productor que marcó "menos riesgo" y "más ingreso", para los ejemplos.
- `VARIACION_PRECIO`: para el paso 7, cuánto se mueve el precio para ver si el orden cambia.

In [2]:
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
DATOS = RAIZ / "backend" / "Datos"
RUTA_ESCENARIOS = DATOS / "escenarios_rotacion.csv"
RUTA_PRECIOS = DATOS / "precios_costos.csv"
RUTA_EXPLICACIONES = DATOS / "front" / "explicaciones.json"

PRIORIDADES = ["income", "risk", "soil", "water"]          # los ids del front (web/src/next/types.ts)
OPCIONES = ["actual", "rotado", "continuo"]
FRACCION_PEORES = 0.10
SUELO_REGLA = {"rotado": 0.9, "actual": 0.55, "continuo": 0.15}   # igual que scripts/build_next_json.py
FACTOR_CONFIANZA = {"high": 1.0, "medium": 1.0, "low": 0.75}
PESOS_EJEMPLO = {"risk": 0.5, "income": 0.5}
VARIACION_PRECIO = 0.20
CONDADO_EJEMPLO = "19169"   # Story, Iowa

HAY_DATOS = RUTA_ESCENARIOS.exists() and RUTA_PRECIOS.exists()
print("¿Están los escenarios del punto 2 y los precios?", HAY_DATOS)
if not HAY_DATOS:
    print("Falta correr punto2_rotaciones.ipynb (maíz y soja) y scripts/build_precios_costos.py. La notebook queda en espera.")

¿Están los escenarios del punto 2 y los precios? True


## 2. Cargar escenarios, precios y confianza

- **Escenarios** (punto 2): una fila por condado, cultivo, opción y año de clima, con el rinde predicho `pred` (bu/ac). Se usa tal cual.
- **Precios y costos** (`precios_costos.csv`): precio recibido de NASS (mediana 2021–2025) y costo operativo del ERS, por estado y cultivo. El costo es el mismo para las tres opciones: el ERS no separa por rotación.
- **Confianza** (punto 3, `explicaciones.json`): el nivel de cada opción, para el `FACTOR_CONFIANZA`.

In [3]:
if HAY_DATOS:
    esc = pd.read_csv(RUTA_ESCENARIOS, dtype={"fips": str})
    precios = pd.read_csv(RUTA_PRECIOS, dtype={"fips_estado": str}).dropna(subset=["precio_usd_bu", "costo_operativo_usd_ac"])
    precios = precios.set_index(["fips_estado", "cultivo"])[["precio_usd_bu", "costo_operativo_usd_ac"]]

    confianza = {}
    if RUTA_EXPLICACIONES.exists():
        ex = json.loads(RUTA_EXPLICACIONES.read_text(encoding="utf-8"))
        confianza = {(f, c, r): d["confidence"]["level"]
                     for f, cultivos in ex["counties"].items() for c, opciones in cultivos.items() for r, d in opciones.items()}

    n = esc.groupby(["fips", "cultivo", "rotacion"]).size()
    print(f"Escenarios: {esc['fips'].nunique()} condados, {len(n)} combinaciones condado × cultivo × opción")
    print(f"Escenarios por combinación: mediana {n.median():.0f}, mínimo {n.min()} | con menos de 10: {(n < 10).sum()}")
    print(f"Precios y costos de {precios.index.get_level_values(0).nunique()} estados | confianza de {len(confianza)} opciones")
else:
    print("En espera de los datos.")

Escenarios: 948 condados, 4341 combinaciones condado × cultivo × opción
Escenarios por combinación: mediana 15, mínimo 3 | con menos de 10: 25
Precios y costos de 11 estados | confianza de 4341 opciones


## 3. Cada prioridad se vuelve una cuenta sobre los escenarios

Para cada opción, cada año de clima se pasa a plata:

> ingreso del año = rinde predicho × precio − costo operativo   ($/acre)

Con los 15 años (uno por clima) salen:

| Métrica | Qué es | Prioridad |
|---|---|---|
| `ingreso_medio_usd_ac` | El promedio de los años | Más ingreso |
| `cvar10_usd_ac` | El promedio de los **peores** años (el 10 %, o sea 2 de 15). Se llama CVaR: "cuánto te queda en un año malo" | Menos riesgo |
| `p_perdida` | En qué parte de los años el ingreso no cubre el costo | Menos riesgo |
| `peor_anio_usd_ac` | El peor año solo | Para mostrar |
| `anios_peores` | Qué años de clima son los peores (el front los puede hacer clickeables) | Para mostrar |

In [4]:
def calcular_metricas(grupo, precio, costo):
    ingreso = grupo["pred"] * precio - costo
    k = max(1, math.ceil(len(ingreso) * FRACCION_PEORES))
    peores = ingreso.nsmallest(k)
    return pd.Series({
        "n_escenarios": len(ingreso),
        "ingreso_medio_usd_ac": ingreso.mean(),
        "cvar10_usd_ac": peores.mean(),
        "peor_anio_usd_ac": ingreso.min(),
        "p_perdida": (ingreso < 0).mean(),
        "anios_peores": ";".join(str(int(a)) for a in grupo.loc[peores.index, "anio_clima"].sort_values()),
    })


if HAY_DATOS:
    filas = []
    for (fips, cultivo, rotacion), g in esc.groupby(["fips", "cultivo", "rotacion"]):
        clave = (fips[:2], cultivo)
        if clave not in precios.index:
            continue
        precio, costo = precios.loc[clave]
        m = calcular_metricas(g, precio, costo)
        filas.append({"fips": fips, "cultivo": cultivo, "rotacion": rotacion, "precio_usd_bu": precio, "costo_usd_ac": costo, **m})
    metricas = pd.DataFrame(filas)
    sin_precio = esc["fips"].str[:2].nunique() - metricas["fips"].str[:2].nunique()
    print(f"Métricas de {len(metricas)} opciones | estados sin precio o costo: {sin_precio}")
    display(metricas.groupby(["cultivo", "rotacion"])[["ingreso_medio_usd_ac", "cvar10_usd_ac", "p_perdida"]].median().round(2))
else:
    print("En espera de los datos.")

Métricas de 4341 opciones | estados sin precio o costo: 0


ingreso_medio_usd_ac  cvar10_usd_ac  p_perdida
cultivo rotacion                                                
maiz    actual                  356.12         246.88        0.0
        continuo                298.78         185.82        0.0
        rotado                  445.99         332.90        0.0
soja    actual                  424.71         367.15        0.0
        continuo                409.90         351.35        0.0
        rotado                  485.74         427.19        0.0

## 4. De métricas a puntajes 0–1 (el formato que ya lee el front)

El front (`web/src/next/suggest.ts`) recibe para cada opción un puntaje de 0 a 1 por prioridad (`scores`) y ordena por el promedio de las prioridades marcadas. Acá se arman esos puntajes, **comparando las opciones del mismo condado y cultivo** (la mejor queda en 1, la peor en 0):

- **income:** según `ingreso_medio_usd_ac`.
- **risk:** el promedio de dos partes: el CVaR (más es mejor) y la probabilidad de pérdida (menos es mejor). Si una parte empata entre las opciones (la probabilidad de pérdida suele ser 0 en las tres), se usa solo la otra.
- **soil:** la regla agronómica `SUELO_REGLA`.
- **water:** vacío. No hay un dato que separe las opciones en agua (la ET de NLDAS es del condado, igual para las tres).

Si todas las opciones empatan en algo (o el condado tiene una sola opción), ese puntaje queda en 0,5 para todas. Después se aplica el `FACTOR_CONFIANZA` a income y risk.

In [5]:
def escalar(serie, mas_es_mejor=True, empate=0.5):
    """0..1 entre las opciones de un mismo condado y cultivo. Con empate (o una sola opción), 'empate'."""
    lo, hi = serie.min(), serie.max()
    if not np.isfinite(lo) or hi - lo < 1e-9:
        return pd.Series(empate, index=serie.index)
    z = (serie - lo) / (hi - lo)
    return z if mas_es_mejor else 1 - z


def calcular_puntajes(m):
    m = m.copy()
    grupos = m.groupby(["fips", "cultivo"])
    m["score_income"] = grupos["ingreso_medio_usd_ac"].transform(escalar)
    cvar = grupos["cvar10_usd_ac"].transform(lambda s: escalar(s, empate=np.nan))
    perdida = grupos["p_perdida"].transform(lambda s: escalar(s, mas_es_mejor=False, empate=np.nan))
    # Si una de las dos partes empata (casi siempre la probabilidad de pérdida, que suele ser 0 en las tres opciones),
    # el riesgo se mide solo con la otra; si empatan las dos, 0,5
    m["score_risk"] = pd.concat([cvar, perdida], axis=1).mean(axis=1, skipna=True).fillna(0.5)
    m["score_soil"] = m["rotacion"].map(SUELO_REGLA)
    m["score_water"] = np.nan
    m["confianza"] = [confianza.get((f, c, r), "medium") for f, c, r in zip(m["fips"], m["cultivo"], m["rotacion"])]
    m["factor_confianza"] = m["confianza"].map(FACTOR_CONFIANZA)
    for p in ("income", "risk"):
        m[f"score_{p}"] = m[f"score_{p}"] * m["factor_confianza"]
    return m


if HAY_DATOS:
    puntajes = calcular_puntajes(metricas)
    print("Opciones con confianza baja (puntaje achicado):", (puntajes["confianza"] == "low").sum())
    display(puntajes[["score_income", "score_risk", "score_soil"]].describe().round(2))
else:
    print("En espera de los datos.")

Opciones con confianza baja (puntaje achicado): 529


,score_income,score_risk,score_soil
count,4341.00,4341.00,4341.00
mean,0.57,0.57,0.54
std,0.40,0.40,0.28
min,0.00,0.00,0.15
25%,0.00,0.00,0.15
50%,0.70,0.70,0.55
75%,0.96,0.97,0.90
max,1.00,1.00,0.90


## 5. Ejemplo: Story (Iowa), maíz, con "menos riesgo" y "más ingreso"

El puntaje total es el promedio pesado de los puntajes de las prioridades. Con los chips del front, cada prioridad marcada pesa lo mismo.

In [6]:
def ordenar(p, fips, cultivo, pesos):
    d = p[(p["fips"] == fips) & (p["cultivo"] == cultivo)].set_index("rotacion")
    usados = {k: v for k, v in pesos.items() if d[f"score_{k}"].notna().any()}
    total = sum(usados.values())
    d = d.assign(puntaje=sum(d[f"score_{k}"] * v / total for k, v in usados.items()))
    return d.sort_values("puntaje", ascending=False)


if HAY_DATOS and CONDADO_EJEMPLO in set(puntajes["fips"]):
    ej = ordenar(puntajes, CONDADO_EJEMPLO, "maiz", PESOS_EJEMPLO)
    display(ej[["n_escenarios", "ingreso_medio_usd_ac", "cvar10_usd_ac", "peor_anio_usd_ac", "p_perdida", "anios_peores",
                "score_income", "score_risk", "score_soil", "puntaje"]].round(2))
else:
    print("En espera de los datos.")

,n_escenarios,ingreso_medio_usd_ac,cvar10_usd_ac,peor_anio_usd_ac,p_perdida,anios_peores,score_income,score_risk,score_soil,puntaje
rotacion,,,,,,,,,,
rotado,15,551.88,460.20,400.73,0.0,2010;2012,1.00,1.00,0.90,1.00
actual,15,498.89,407.18,347.71,0.0,2010;2012,0.65,0.65,0.55,0.65
continuo,15,399.38,307.61,248.14,0.0,2010;2012,0.00,0.00,0.15,0.00


## 6. De "lo que le importa" a pesos: tres formas

| Forma | Cómo se pregunta | Pesos |
|---|---|---|
| **Chips** (lo de hoy) | Marca las prioridades | Cada una pesa lo mismo |
| **Ordenar** | Arrastra las prioridades de más a menos importante | Decrecientes: con 3, 0,50 / 0,33 / 0,17 (pesos por suma de rangos) |
| **Comparar de a pares** | "¿Preferís $40 más en un año típico o $40 menos de pérdida en un año malo?", 4 o 5 veces | Se ajustan con una **regresión logística**: el peso de cada prioridad es cuánto pesa su diferencia en cada elección (análisis conjunto) |

La regresión logística aprende de las respuestas del productor, no de los datos del campo. Abajo va **un ejemplo de cómo se usaría**, con las respuestas de un productor imaginario muy cuidadoso con los años malos. No es un resultado: es una demostración del método.

In [7]:
def pesos_chips(marcadas):
    return {p: 1 / len(marcadas) for p in marcadas}


def pesos_orden(ordenadas):
    """Pesos por suma de rangos: la primera vale n, la segunda n−1… normalizado a 1."""
    n = len(ordenadas)
    total = n * (n + 1) / 2
    return {p: (n - i) / total for i, p in enumerate(ordenadas)}


def pesos_pares(respuestas, prioridades=("income", "risk")):
    """respuestas: lista de (diferencias A − B por prioridad, eligió A). Diferencias en $/acre.
    Ajusta una logística sin intercepto (de a pares: solo importan las diferencias) y normaliza los coeficientes."""
    X = np.array([[d[p] for p in prioridades] for d, _ in respuestas], dtype=float)
    y = np.array([int(eligio_a) for _, eligio_a in respuestas])
    X = X / np.abs(X).max()                           # una sola escala: las dos diferencias están en $/acre
    modelo = LogisticRegression(fit_intercept=False, C=10.0).fit(X, y)
    coef = np.clip(modelo.coef_[0], 0, None)          # un peso negativo no tiene sentido como prioridad
    return dict(zip(prioridades, coef / coef.sum())) if coef.sum() > 0 else pesos_chips(list(prioridades))


print("Chips (riesgo, ingreso):", pesos_chips(["risk", "income"]))
print("Ordenar (riesgo > ingreso > suelo):", {k: round(v, 2) for k, v in pesos_orden(["risk", "income", "soil"]).items()})

# EJEMPLO, no datos: un productor que en 5 preguntas elige casi siempre la opción con mejor año malo.
# Cada pregunta: (diferencias A − B en $/acre de ingreso típico y de año malo, eligió A).
respuestas_ejemplo = [
    ({"income": +40, "risk": -40}, False),
    ({"income": +80, "risk": -40}, False),
    ({"income": +120, "risk": -40}, True),
    ({"income": -20, "risk": +40}, True),
    ({"income": +40, "risk": -10}, True),
]
print("Comparar de a pares (ejemplo):", {k: round(v, 2) for k, v in pesos_pares(respuestas_ejemplo).items()})

Chips (riesgo, ingreso): {'risk': 0.5, 'income': 0.5}
Ordenar (riesgo > ingreso > suelo): {'risk': 0.5, 'income': 0.33, 'soil': 0.17}
Comparar de a pares (ejemplo): {'income': np.float64(0.3), 'risk': np.float64(0.7)}


## 7. Validación: ¿las prioridades cambian el orden?

No hay un "dato real" del orden correcto: cada productor pesa distinto. Lo que sí se puede medir es si la capa de decisión **responde** a las prioridades y al precio. Para cada condado y cultivo con más de una opción se ve qué opción queda primera con distintos pesos:

- solo ingreso, solo riesgo, solo suelo;
- chips con las tres;
- ordenar riesgo > ingreso > suelo;
- solo ingreso y solo riesgo con el precio ±20 %.

Y se compara con **cómo puntuaba el front antes** (riesgo según el p10 del rinde e ingreso según la mediana): ahí riesgo, rinde e ingreso ordenaban casi igual, así que las prioridades casi nunca cambiaban la opción que queda primera.

In [8]:
def primera(p, pesos):
    usados = {k: v for k, v in pesos.items() if p[f"score_{k}"].notna().any()}
    total = sum(usados.values())
    q = p.assign(puntaje=sum(p[f"score_{k}"] * v / total for k, v in usados.items()))
    return q.sort_values("puntaje", ascending=False).groupby(["fips", "cultivo"])["rotacion"].first()


if HAY_DATOS:
    varias = puntajes.groupby(["fips", "cultivo"])["rotacion"].transform("nunique") > 1
    base = puntajes[varias]
    escenarios_pesos = {
        "solo ingreso": {"income": 1},
        "solo riesgo": {"risk": 1},
        "solo suelo": {"soil": 1},
        "chips: las tres": pesos_chips(["income", "risk", "soil"]),
        "orden: riesgo > ingreso > suelo": pesos_orden(["risk", "income", "soil"]),
    }
    ganadoras = pd.DataFrame({k: primera(base, w) for k, w in escenarios_pesos.items()})

    for signo, nombre in [(1 - VARIACION_PRECIO, "precio −20 %"), (1 + VARIACION_PRECIO, "precio +20 %")]:
        m2 = metricas.assign(precio_usd_bu=metricas["precio_usd_bu"] * signo)
        filas2 = []
        for (f, c, r), g in esc.groupby(["fips", "cultivo", "rotacion"]):
            if (f[:2], c) in precios.index:
                pr, co = precios.loc[(f[:2], c)]
                filas2.append({"fips": f, "cultivo": c, "rotacion": r, **calcular_metricas(g, pr * signo, co)})
        p2 = calcular_puntajes(pd.DataFrame(filas2))
        p2 = p2[p2.groupby(["fips", "cultivo"])["rotacion"].transform("nunique") > 1]
        ganadoras[f"solo riesgo, {nombre}"] = primera(p2, {"risk": 1})

    # Cómo puntuaba el front antes: riesgo = p10 del rinde, ingreso = mediana del ingreso (mismo orden que el rinde)
    resumen = pd.read_csv(DATOS / "escenarios_rotacion_resumen.csv", dtype={"fips": str})
    viejo = resumen.merge(base[["fips", "cultivo", "rotacion"]], on=["fips", "cultivo", "rotacion"])
    viejo_riesgo = viejo.sort_values("p10_clima", ascending=False).groupby(["fips", "cultivo"])["rotacion"].first()
    viejo_ingreso = viejo.sort_values("mediana_clima", ascending=False).groupby(["fips", "cultivo"])["rotacion"].first()

    n_casos = len(ganadoras)
    cambia_ri = (ganadoras["solo riesgo"] != ganadoras["solo ingreso"]).sum()
    cambia_viejo = (viejo_riesgo != viejo_ingreso).sum()
    cambia_alguna = (ganadoras[list(escenarios_pesos)].nunique(axis=1) > 1).sum()
    cambia_precio = (ganadoras["solo riesgo, precio −20 %"] != ganadoras["solo riesgo, precio +20 %"]).sum()

    print(f"Condado × cultivo con más de una opción: {n_casos}")
    print(f"Riesgo e ingreso eligen distinta opción primera: {cambia_ri} ({cambia_ri / n_casos:.0%}). "
          f"Con la forma vieja del front: {cambia_viejo} ({cambia_viejo / n_casos:.0%})")
    print(f"Alguna combinación de prioridades cambia la opción primera: {cambia_alguna} ({cambia_alguna / n_casos:.0%})")
    print(f"Con 'solo riesgo', mover el precio ±20 % cambia la primera en: {cambia_precio} ({cambia_precio / n_casos:.0%})")
    display(pd.DataFrame({k: ganadoras[k].value_counts() for k in ganadoras}).fillna(0).astype(int).T)
else:
    print("En espera de los datos.")

Condado × cultivo con más de una opción: 1244
Riesgo e ingreso eligen distinta opción primera: 6 (0%). Con la forma vieja del front: 0 (0%)
Alguna combinación de prioridades cambia la opción primera: 119 (10%)
Con 'solo riesgo', mover el precio ±20 % cambia la primera en: 12 (1%)


,actual,rotado
solo ingreso,118,1126
solo riesgo,114,1130
solo suelo,0,1244
chips: las tres,44,1200
orden: riesgo > ingreso > suelo,98,1146
"solo riesgo, precio −20 %",119,1125
"solo riesgo, precio +20 %",117,1127


## 8. Guardar

| Archivo | Qué tiene | Para quién |
|---|---|---|
| `backend/Datos/decision_opciones.csv` | Una fila por condado × cultivo × opción: `n_escenarios`, `precio_usd_bu`, `costo_usd_ac`, `ingreso_medio_usd_ac`, `cvar10_usd_ac`, `peor_anio_usd_ac`, `p_perdida`, `anios_peores`, `confianza`, `factor_confianza`, `score_income`, `score_risk`, `score_soil`, `score_water` | `scripts/build_next_json.py` (llena `scores` y las razones de riesgo e ingreso) |
| `backend/Datos/decision_sensibilidad.csv` | Qué opción queda primera con cada combinación de pesos y precio, por condado y cultivo | Revisión y pitch |

In [9]:
if HAY_DATOS:
    salida = puntajes.round({"precio_usd_bu": 2, "costo_usd_ac": 2, "ingreso_medio_usd_ac": 1, "cvar10_usd_ac": 1,
                             "peor_anio_usd_ac": 1, "p_perdida": 3, "score_income": 3, "score_risk": 3, "score_soil": 2})
    salida.to_csv(DATOS / "decision_opciones.csv", index=False)
    ganadoras.reset_index().to_csv(DATOS / "decision_sensibilidad.csv", index=False)
    print(f"Guardados decision_opciones.csv ({len(salida)} filas) y decision_sensibilidad.csv ({len(ganadoras)} filas)")
else:
    print("En espera de los datos.")

Guardados decision_opciones.csv (4341 filas) y decision_sensibilidad.csv (1244 filas)


## Próximos pasos

- [ ] Que el front pase a pesos por orden o por comparación de a pares (`suggest.ts`: hoy promedia las prioridades marcadas con pesos iguales).
- [ ] Costo por rotación: el maíz continuo lleva más nitrógeno; con un costo propio por opción el orden por ingreso puede cambiar.
- [ ] "Ahorrar agua": falta un dato que separe las opciones (la ET de NLDAS es del condado).
- [ ] Salud del suelo con datos (carbono de SSURGO o diversidad de la rotación) en vez de la regla fija.